# PocketAcc – demo notebook (fine-tuned Qwen2-VL-2B)

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On**.

| When | What to do | Time |
|---|---|---|
| **Tonight** | *Save Version → Save & Run All*. Then open that version's **Output** tab and download **`demo_results.zip`**. | ~25 min |
| **Live demo** | Start a session 10 min early. Run **cell 1** and **cell 2**, then go to **Part B** (and Part C if you want the web app link). | model loads in ~2 min, ~2–3 min per page |

The trained adapter comes from the GitHub repo (`adapters/lora_r8`, 8.7 MB), so no extra input is needed.
Part B and C are skipped automatically during *Save Version*.

In [ ]:
# 1. code, system tools and Python packages (~3 min)
BRANCH = "demo-ready"
import os
if not os.path.exists("/kaggle/working/pocketacc"):
    !git clone -q -b $BRANCH https://github.com/anshace07/PocketAcc.git /kaggle/working/pocketacc
else:
    !git -C /kaggle/working/pocketacc pull -q
%cd /kaggle/working/pocketacc
!git log --oneline -3
!nvidia-smi -L
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip -q install -r requirements.txt -r requirements-gpu.txt -r requirements-app.txt
assert os.path.exists("adapters/lora_r8/adapter_model.safetensors"), "trained adapter missing from the repo"
print("ready")

In [ ]:
# 2. helpers: run a command and STOP if it fails; run two commands side by side on the two GPUs
import subprocess, sys, time
from pathlib import Path
sys.path[:0] = ["src", "scripts"]
Path("logs").mkdir(exist_ok=True)
N_GPU = len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines())
LIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") != "Batch"      # False during Save Version
print("GPUs:", N_GPU, "| live session:", LIVE)

def sh(cmd, gpu=0):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

def parallel(jobs):
    if N_GPU < 2:
        for cmd in jobs.values():
            sh(cmd)
        return
    procs = {}
    for i, (name, cmd) in enumerate(jobs.items()):
        procs[name] = subprocess.Popen(cmd, shell=True, stdout=open(f"logs/{name}.log", "w"), stderr=subprocess.STDOUT,
                                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % N_GPU)))
        print(f"started {name} on GPU {i % N_GPU}", flush=True)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(60)
        for name, p in procs.items():
            lines = [l for l in open(f"logs/{name}.log", errors="ignore").read().splitlines() if l.strip()]
            state = "running" if p.poll() is None else f"finished (exit {p.returncode})"
            print(f"[{time.strftime('%H:%M')}] {name}: {state} | {lines[-1][:150] if lines else ''}", flush=True)
    for name in procs:
        print(f"\n----- {name} -----\n" + "\n".join(open(f"logs/{name}.log", errors="ignore").read().splitlines()[-12:]))
    failed = [n for n, p in procs.items() if p.returncode]
    if failed:
        raise RuntimeError(f"FAILED: {failed}")

## Part A – save the fine-tuned model's results on the demo statements (~20 min, runs in *Save Version*)

The demo statements (`demo/samples/`) are from **PocketAcc Demo Bank** – a made-up bank layout the model never saw.
Statement A is also photographed with light, medium and heavy damage. Every file has an answer key, so we can say
exactly how many transactions were read right. Tesseract OCR runs on the same files for comparison.

In [ ]:
S = "demo/samples/"
parallel({
    "vlm_gpu0": f"python scripts/run_demo.py --backend vlm --files {S}A_salaried_statement.pdf {S}A_page1_photo_light.jpg {S}A_page1_photo_medium.jpg",
    "vlm_gpu1": f"python scripts/run_demo.py --backend vlm --files {S}A_page1_photo_heavy.jpg {S}B_shop_statement.pdf",
})
sh("python scripts/run_demo.py --backend tesseract")

In [ ]:
# side-by-side: fine-tuned model vs Tesseract on every demo file
import json, glob
import pandas as pd
rows = []
for f in sorted(glob.glob("demo/results/*.json")):
    r = json.load(open(f)); s = r["summary"]; k = r.get("answer_key") or {}
    rows.append({"file": r["file"], "reader": r["saved"]["reader"], "rows read": s["transactions"],
                 "answer key rows": k.get("answer_rows"), "date+amounts exactly right": k.get("numbers_right"),
                 "also exact text": k.get("fully_right"), "trust %": s["trust_score"],
                 "auto-corrected": s["auto_corrected"], "needs review": s["needs_review"], "unusual": s["anomalies"],
                 "seconds": r["seconds"]})
pd.set_option("display.width", 250)
display(pd.DataFrame(rows))

In [ ]:
!cd demo && zip -qr /kaggle/working/demo_results.zip results && ls -lh /kaggle/working/demo_results.zip
print("Download demo_results.zip from the Output tab of this version.")

## Part B – LIVE: the fine-tuned model reads a statement in front of you

Run cell 1 and cell 2 first. Loading takes ~2 min; then each page takes ~2–3 min on a T4 (talk through the pipeline while it reads).
Change `FILE` to any PDF or photo – e.g. `demo/samples/B_shop_statement.pdf`, or a file you uploaded
(*File → Upload Input*, it appears under `/kaggle/input/...`).

In [ ]:
if LIVE:
    import demo_live
    demo_live.load()

In [ ]:
FILE = "demo/samples/A_page1_photo_heavy.jpg"
if LIVE:
    result = demo_live.show(FILE)          # compare_tesseract=False to skip the OCR comparison

## Part C – optional: the full web app with the fine-tuned model, as a public link

Starts the PocketAcc dashboard on the second GPU and prints a `https://….trycloudflare.com` link that opens on any laptop or phone.
Upload a PDF/photo or click a sample; *Saved results* shows Part A's results instantly. The link works while this session runs.
If no link appears, use Part B or the offline results page instead.

In [ ]:
if LIVE:
    import start_public_app
    URL = start_public_app.start()